In [27]:
import pandas as pd
import plotly.express as px

# Cargar el conjunto de datos desde la raíz del proyecto
df = pd.read_csv("../vehicles_us.csv")

# Renombrar columnas a minúsculas para un manejo estandarizado
df.columns = df.columns.str.lower()

# Extraer la marca del vehículo (primera palabra del modelo) para análisis futuros
df['brand'] = df['model'].apply(lambda x: x.split()[0].title())

print("Dimensiones del dataset:", df.shape)
df.head()

Dimensiones del dataset: (51525, 14)


,price,model_year,model,condition,cylinders,fuel,odometer,transmission,type,paint_color,is_4wd,date_posted,days_listed,brand
0,9400,2011.0,bmw x5,good,6.0,gas,145000.0,automatic,SUV,NaN,1.0,2018-06-23,19,Bmw
1,25500,NaN,ford f-150,good,6.0,gas,88705.0,automatic,pickup,white,1.0,2018-10-19,50,Ford
2,5500,2013.0,hyundai sonata,like new,4.0,gas,110000.0,automatic,sedan,red,NaN,2019-02-07,79,Hyundai
3,1500,2003.0,ford f-150,fair,8.0,gas,NaN,automatic,pickup,NaN,NaN,2019-03-22,9,Ford
4,14900,2017.0,chrysler 200,excellent,4.0,gas,80903.0,automatic,sedan,black,NaN,2019-04-02,28,Chrysler


In [28]:
# Revisar la cantidad exacta de valores nulos por columna
print("Valores ausentes iniciales:")
print(df.isna().sum())

Valores ausentes iniciales:
price               0
model_year       3619
model               0
condition           0
cylinders        5260
fuel                0
odometer         7892
transmission        0
type                0
paint_color      9267
is_4wd          25953
date_posted         0
days_listed         0
brand               0
dtype: int64


In [29]:
# Vamos a usar el dato mas común para rellenar
# Año del modelo:
df['model_year'] = df.groupby('model')['model_year'].transform(
    lambda x: x.fillna(x.mode()[0] if not x.mode().empty else 2010)
).astype(int)

# Cilindros: Rellenar con la moda de cilindros de ese modelo específico
df['cylinders'] = df.groupby('model')['cylinders'].transform(
    lambda x: x.fillna(x.mode()[0] if not x.mode().empty else 6)).astype(int)

#Vamos a usar la mediana para rellenar Nan:
# Odómetro (Kilometraje): según el año y modelo del auto
df['odometer'] = df.groupby(['model_year', 'model'])['odometer'].transform(
    lambda x: x.fillna(x.median() if not x.isna().all() else df['odometer'].median())
).astype(int)

# Tracción de cuatro ruedas: Si es NaN asumimos que no tiene (0 = No, 1 = Sí)
df['is_4wd'] = df['is_4wd'].fillna(0).astype(int)

# Convertir la columna de fecha a tipo datetime real
df['date_posted'] = pd.to_datetime(df['date_posted'], errors='coerce')

# Confirmar que ya no quedan valores nulos en el dataset
print("Valores ausentes después de la limpieza:")
print(df.isna().sum())


Valores ausentes después de la limpieza:
price              0
model_year         0
model              0
condition          0
cylinders          0
fuel               0
odometer           0
transmission       0
type               0
paint_color     9267
is_4wd             0
date_posted        0
days_listed        0
brand              0
dtype: int64


,price,model_year,model,condition,cylinders,fuel,odometer,transmission,type,paint_color,is_4wd,date_posted,days_listed,brand
count,51525.000000,51525.000000,51525,51525,51525.000000,51525,51525.000000,51525,51525,42258,51525.000000,51525,51525.00000,51525
unique,NaN,NaN,100,6,NaN,5,NaN,3,13,12,NaN,NaN,NaN,19
top,NaN,NaN,ford f-150,excellent,NaN,gas,NaN,automatic,SUV,white,NaN,NaN,NaN,Ford
freq,NaN,NaN,2796,24773,NaN,47288,NaN,46902,12405,10029,NaN,NaN,NaN,12672
mean,12132.464920,2009.871033,NaN,NaN,6.121494,NaN,115350.970267,NaN,NaN,NaN,0.496303,2018-10-25 01:57:46.270742,39.55476,NaN
min,1.000000,1908.000000,NaN,NaN,3.000000,NaN,0.000000,NaN,NaN,NaN,0.000000,2018-05-01 00:00:00,0.00000,NaN
25%,5000.000000,2007.000000,NaN,NaN,4.000000,NaN,72106.000000,NaN,NaN,NaN,0.000000,2018-07-29 00:00:00,19.00000,NaN
50%,9000.000000,2011.000000,NaN,NaN,6.000000,NaN,113703.000000,NaN,NaN,NaN,0.000000,2018-10-25 00:00:00,33.00000,NaN
75%,16839.000000,2014.000000,NaN,NaN,8.000000,NaN,153000.000000,NaN,NaN,NaN,1.000000,2019-01-21 00:00:00,53.00000,NaN
max,375000.000000,2019.000000,NaN,NaN,12.000000,NaN,990000.000000,NaN,NaN,NaN,1.000000,2019-04-19 00:00:00,271.00000,NaN


In [31]:
# Gráfico interactivo para analizar la concentración de precios del mercado
fig_price = px.histogram(df, x="price", nbins=50, 
                         title="Distribución de Precios de Venta de Vehículos",
                         labels={"price": "Precio ($)"},
                         color_discrete_sequence=['#1f77b4'])
fig_price.show()

In [33]:
# Gráfico interactivo para analizar el impacto del desgaste mecánico en el valor de reventa
fig_scatter = px.scatter(df, x="odometer", y="price", color="condition",
                         title="Análisis de Depreciación: Precio vs Kilometraje según Condición",
                         labels={"odometer": "Kilometraje (Millas)", "price": "Precio ($)"},
                         opacity=0.5)
fig_scatter.show()

In [34]:
# Conocer qué tipo de cajas de cambios predominan en la oferta de EE.UU.
fig_trans = px.histogram(df, x="transmission", 
                         title="Volumen de Anuncios por Tipo de Transmisión",
                         labels={"transmission": "Tipo de Transmisión"},
                         color="transmission")
fig_trans.show()

Concluciones 

En este análisis exploratorio podemos ver que la mayoría de los autos se concentran en rangos por debajo de los 15,000 USD, mientras que los precios elevados corresponden a casos aislados de vehículos de lujo. Desto pomos decir que se estr tratando de llegar a un publico general que pueda adquirir estos autos.
Al observar la relación entre precio y kilometraje, se confirma una tendencia clara a medida que aumenta el odómetro, el valor del vehículo disminuye de manera significativa. Sin embargo, la variable de condición resulta determinante, ya que los autos en estado “excelente” o “como nuevo” logran mantener un precio significativamente mayor incluso con un kilometraje elevado, en contraste con aquellos en condición “buena” o “regular”, que pierden valor más rápidamente.